# Repository OSS Health Diagnosis

특정 GitHub repository URL을 입력하면 다음 결과를 생성한다.

1. 저장된 최종 모델의 `predict_proba`를 사용한 총괄 OSS health score
2. 제안서의 5개 평가 차원별 score
3. 각 차원별 강점, 약점, 해석 문장
4. backend에서 사용할 수 있는 진단 함수 형태의 로직

총괄 점수는 binary threshold 적용 전 모델 확률을 0~100점으로 변환한 값이다.

차원별 점수는 모델 확률이 아니라, reference dataset 대비 feature percentile을 이용해 계산한다. 이 방식은 특정 repository가 학습 데이터의 repository들과 비교했을 때 각 평가 차원에서 어느 정도 위치에 있는지 보여준다.


## Imports and Paths

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import joblib

from data.extract_github_repo import build_repo_dataframe
from features.build_features import build_all_features

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data/main/dataset.csv"
MODEL_PATH = BASE_DIR / "models/oss_health_best_model.joblib"
FEATURE_PATH = BASE_DIR / "models/oss_health_best_features.json"
METADATA_PATH = BASE_DIR / "models/oss_health_model_metadata.json"
OUTPUT_DIR = BASE_DIR / "outputs/3_repository_diagnosis"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH = Path("/Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/data/main/dataset.csv")
if not MODEL_PATH.exists():
    MODEL_PATH = Path("/Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/models/oss_health_best_model.joblib")
if not FEATURE_PATH.exists():
    FEATURE_PATH = Path("/Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/models/oss_health_best_features.json")
if not METADATA_PATH.exists():
    METADATA_PATH = Path("/Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/models/oss_health_model_metadata.json")


## Load Model Artifacts

In [2]:
model = joblib.load(MODEL_PATH)

with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    model_features = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    model_metadata = json.load(f)

print("Model path:", MODEL_PATH)
print("Model type:", model_metadata.get("best_model_name"))
print("Target:", model_metadata.get("target_col"))
print("Number of model features:", len(model_features))
print("Holdout ROC-AUC:", model_metadata.get("tuned_holdout_metrics", {}).get("roc_auc"))


Model path: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/models/oss_health_best_model.joblib
Model type: LogisticRegression
Target: new_label
Number of model features: 101
Holdout ROC-AUC: 0.9819976771196284


## Feature Engineering Utilities

`2_MODEL.ipynb`에서 학습할 때 사용한 engineered feature를 동일하게 재현한다.

In [3]:
def minmax_series(values):
    values = values.astype(float)
    min_value = values.min()
    max_value = values.max()

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.5, index=values.index)

    return (values - min_value) / (max_value - min_value)


def to_numeric_columns(data, columns):
    out = data.copy()

    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def add_engineered_features(data):
    out = data.copy()
    eps = 1e-9

    numeric_cols = [col for col in out.columns if col not in ["repo_name", "dominant_event_type", "error"]]
    out = to_numeric_columns(out, numeric_cols)

    out["is_recently_pushed_30d"] = (out["last_push_recency_days"] <= 30).astype(int)
    out["is_recently_updated_90d"] = (out["last_update_recency_days"] <= 90).astype(int)
    out["is_recently_pushed_90d"] = (out["last_push_recency_days"] <= 90).astype(int)
    out["is_stale_365d"] = (out["last_push_recency_days"] > 365).astype(int)

    out["push_update_consistency"] = 1 / (1 + out["update_push_gap_days"])
    out["freshness_score"] = (
        1 / (1 + out["last_push_recency_days"])
        + 1 / (1 + out["last_update_recency_days"])
    ) / 2

    out["bus_factor_risk"] = (
        out["top1_contribution_share"] + out["contribution_gini"]
    ) / 2
    out["distributed_contribution_score"] = (
        out["contribution_entropy"] * (1 - out["top1_contribution_share"])
    )
    out["contributor_depth_score"] = (
        np.log1p(out["num_contributors"])
        * np.log1p(out["median_contributions"])
    )

    out["release_maturity_score"] = (
        out["stable_tag_ratio"]
        * out["semver_tag_ratio"]
        * out["latest_tag_is_stable"]
    )
    out["active_release_score"] = (
        np.log1p(out["num_tags"]) * out["tag_release_velocity"]
    )
    out["release_recency_score"] = 1 / (1 + out["deployment_recency_days"])
    out["release_quality_score"] = (
        out["release_maturity_score"]
        + out["release_recency_score"]
        + minmax_series(out["active_release_score"].fillna(0))
    ) / 3

    out["collaboration_event_score"] = (
        out["PullRequestEvent_ratio"]
        + out["IssueCommentEvent_ratio"]
        + out["IssuesEvent_ratio"]
    )
    out["activity_diversity_score"] = (
        out["event_type_entropy"] * out["num_unique_event_types"]
    )
    out["healthy_activity_score"] = (
        out["interaction_ratio"]
        + out["development_ratio"]
        + out["collaboration_event_score"]
    ) / 3
    out["non_external_activity_ratio"] = 1 - out["external_interest_event_ratio"]

    out["adoption_efficiency"] = (
        np.log1p(out["stargazers_count"])
        / np.log1p(out["repo_age_days"] + 1)
    )
    out["issue_burden_score"] = (
        out["open_issues_count"]
        / (np.log1p(out["stargazers_count"]) + eps)
    )
    out["fork_interest_efficiency"] = (
        np.log1p(out["forks_count"])
        / np.log1p(out["repo_age_days"] + 1)
    )

    out["governance_openness_score"] = (
        out["has_issues"]
        + out["has_projects"]
        + out["has_wiki"]
        + out["has_discussions"]
        + out["has_pull_requests"]
    ) / 5
    out["negative_repo_state"] = (
        out["archived"] + out["disabled"]
    ).clip(0, 1)

    out["maintainer_activity_score"] = (
        out["is_recently_pushed_30d"]
        + out["is_recently_updated_90d"]
        + out["push_update_consistency"]
    ) / 3

    out.replace([np.inf, -np.inf], np.nan, inplace=True)

    return out


## Reference Dataset

차원별 점수는 reference dataset 대비 percentile로 계산한다.

예를 들어 특정 repo의 `event_type_entropy`가 reference dataset의 80 percentile 위치에 있으면 해당 feature score는 80점에 가깝게 계산된다. unhealthy 방향 feature는 percentile을 반대로 뒤집는다.

In [4]:
reference_raw = pd.read_csv(DATA_PATH)
reference_df = add_engineered_features(reference_raw)

for feature in model_features:
    if feature not in reference_df.columns:
        reference_df[feature] = np.nan

reference_X = reference_df[model_features].copy()

for col in model_features:
    reference_X[col] = pd.to_numeric(reference_X[col], errors="coerce")

reference_X.shape


(411, 101)

## Five Evaluation Dimensions

제안서의 5개 기준을 현재 feature space에 대응시킨다.

단, 현재 dataset은 GitHub API 기반 feature로 구성되어 있으므로 `코드 품질 및 신뢰성`과 `법적/운영 거버넌스` 중 일부 개념은 proxy feature로 측정한다.

예를 들어 license compliance, vulnerability, CI/test coverage는 현재 feature에 직접 포함되어 있지 않으므로, release practice, issue burden, pull request activity, archived/disabled 여부, governance option을 대체 지표로 사용한다.

In [5]:
dimension_config = {
    "community_activity": {
        "label_ko": "커뮤니티 활성도",
        "core_question": "이 프로젝트는 현재 살아 움직이고 있는가?",
        "concepts": "Activity Volume, Responsiveness, Engagement Quality",
        "features": [
            "num_events",
            "num_unique_event_types",
            "event_type_entropy",
            "recent_event_density",
            "has_IssuesEvent",
            "has_PullRequestEvent",
            "has_IssueCommentEvent",
            "IssuesEvent_ratio",
            "IssueCommentEvent_ratio",
            "PullRequestEvent_ratio",
            "interaction_ratio",
            "development_ratio",
            "collaboration_event_score",
            "activity_diversity_score",
            "healthy_activity_score",
            "non_external_activity_ratio",
            "external_interest_event_ratio",
            "dominant_event_ratio",
        ],
    },
    "sustainability": {
        "label_ko": "지속 가능성",
        "core_question": "이 프로젝트는 앞으로도 유지될 수 있는가?",
        "concepts": "Contributor Structure, Diversity, Activity Stability",
        "features": [
            "num_contributors",
            "total_contributions",
            "median_contributions",
            "contribution_entropy",
            "top1_contribution_share",
            "top5_contribution_share",
            "contribution_gini",
            "top1_contribution_ratio",
            "top3_contribution_ratio",
            "contributors_to_stars_ratio",
            "bus_factor_risk",
            "distributed_contribution_score",
            "contributor_depth_score",
            "last_update_recency_days",
            "last_push_recency_days",
            "update_push_gap_days",
            "is_recently_pushed_30d",
            "is_recently_updated_90d",
            "is_stale_365d",
            "push_update_consistency",
            "freshness_score",
            "maintainer_activity_score",
        ],
    },
    "code_quality_reliability": {
        "label_ko": "코드 품질 및 신뢰성",
        "core_question": "이 프로젝트의 산출물은 믿을 수 있는가?",
        "concepts": "Engineering Practice, Defect Signals, Security Signals",
        "features": [
            "semver_tag_ratio",
            "stable_tag_ratio",
            "latest_tag_is_stable",
            "latest_tag_is_prerelease",
            "prerelease_tag_ratio",
            "release_maturity_score",
            "release_quality_score",
            "has_pull_requests",
            "PullRequestEvent_ratio",
            "development_ratio",
            "open_issues_to_stars_ratio",
            "issues_per_size",
            "issue_burden_score",
            "compiled_ratio",
            "language_entropy",
        ],
    },
    "legal_operational_governance": {
        "label_ko": "법적/운영 거버넌스",
        "core_question": "이 프로젝트는 조직적으로 안전하게 운영되는가?",
        "concepts": "Legal Compliance, Governance Structure",
        "features": [
            "has_issues",
            "has_projects",
            "has_downloads",
            "has_wiki",
            "has_pages",
            "has_discussions",
            "allow_forking",
            "has_pull_requests",
            "governance_openness_score",
            "archived",
            "disabled",
            "negative_repo_state",
        ],
    },
    "project_maturity": {
        "label_ko": "프로젝트 성숙도",
        "core_question": "이 프로젝트는 성숙한 운영 체계를 갖추었는가?",
        "concepts": "Release Engineering, Adoption/Popularity, Lifecycle/Scale",
        "features": [
            "num_tags",
            "num_major_versions",
            "num_minor_versions",
            "tag_release_velocity",
            "num_deployments",
            "has_deployments",
            "num_unique_refs",
            "tag_based_deployment_ratio",
            "deployment_recency_days",
            "release_recency_score",
            "active_release_score",
            "release_quality_score",
            "stargazers_count",
            "subscribers_count",
            "forks_count",
            "network_count",
            "stars_per_repo_age_day",
            "forks_per_repo_age_day",
            "stars_per_size",
            "forks_per_size",
            "adoption_efficiency",
            "fork_interest_efficiency",
            "repo_age_days",
            "repo_size",
        ],
    },
}

negative_score_features = {
    "top1_contribution_share",
    "top5_contribution_share",
    "contribution_gini",
    "top1_contribution_ratio",
    "top3_contribution_ratio",
    "bus_factor_risk",
    "deployment_recency_days",
    "prerelease_tag_ratio",
    "latest_tag_is_prerelease",
    "last_update_recency_days",
    "last_push_recency_days",
    "update_push_gap_days",
    "is_stale_365d",
    "open_issues_to_stars_ratio",
    "issues_per_size",
    "issue_burden_score",
    "dominant_event_ratio",
    "external_interest_event_ratio",
    "archived",
    "disabled",
    "negative_repo_state",
}


## Dimension Scoring Logic

In [6]:
def percentile_score(value, reference_values, higher_is_better=True):
    reference_values = pd.to_numeric(reference_values, errors="coerce").dropna().values.astype(float)

    if len(reference_values) == 0 or pd.isna(value):
        return np.nan

    percentile = (reference_values <= float(value)).mean()

    if not higher_is_better:
        percentile = 1 - percentile

    return float(np.clip(percentile * 100, 0, 100))


def get_feature_weight_table():
    importance_paths = [
        OUTPUT_DIR.parent / "2_model/final_model_importance.csv",
        OUTPUT_DIR.parent / "2_model/final_model_permutation_importance.csv",
        OUTPUT_DIR.parent / "2_model/shap_importance_result.csv",
    ]

    weights = pd.DataFrame({"feature": model_features})

    for path in importance_paths:
        if not path.exists():
            continue

        temp = pd.read_csv(path)

        score_cols = [
            col for col in temp.columns
            if col in [
                "model_importance",
                "permutation_importance_mean",
                "mean_abs_shap",
                "abs_coef",
            ]
        ]

        if not score_cols:
            continue

        score_col = score_cols[0]
        temp = temp[["feature", score_col]].copy()
        temp[score_col] = pd.to_numeric(temp[score_col], errors="coerce").clip(lower=0)
        temp = temp.rename(columns={score_col: f"weight_{score_col}"})
        weights = weights.merge(temp, on="feature", how="left")

    weight_cols = [col for col in weights.columns if col.startswith("weight_")]

    if weight_cols:
        weights["weight"] = weights[weight_cols].fillna(0).mean(axis=1)
    else:
        weights["weight"] = 1.0

    if weights["weight"].sum() <= 0:
        weights["weight"] = 1.0

    return weights[["feature", "weight"]]


feature_weights = get_feature_weight_table()
feature_weight_map = dict(zip(feature_weights["feature"], feature_weights["weight"]))


def score_dimension(repo_features, dimension_key):
    config = dimension_config[dimension_key]
    available_features = [
        feature for feature in config["features"]
        if feature in repo_features.columns and feature in reference_X.columns
    ]

    rows = []

    for feature in available_features:
        value = repo_features[feature].iloc[0]
        higher_is_better = feature not in negative_score_features
        score = percentile_score(value, reference_X[feature], higher_is_better=higher_is_better)
        weight = feature_weight_map.get(feature, 1.0)

        if pd.isna(score):
            continue

        rows.append({
            "dimension": dimension_key,
            "dimension_label": config["label_ko"],
            "feature": feature,
            "raw_value": value,
            "higher_is_better": higher_is_better,
            "feature_score": score,
            "weight": weight,
        })

    detail = pd.DataFrame(rows)

    if detail.empty:
        return np.nan, detail

    effective_weight = detail["weight"].clip(lower=0).replace(0, np.nan)

    if effective_weight.isna().all():
        dimension_score = detail["feature_score"].mean()
    else:
        effective_weight = effective_weight.fillna(effective_weight.median())
        dimension_score = np.average(detail["feature_score"], weights=effective_weight)

    return float(np.clip(dimension_score, 0, 100)), detail


def score_to_grade(score):
    if score >= 85:
        return "Excellent"
    if score >= 70:
        return "Good"
    if score >= 55:
        return "Moderate"
    if score >= 40:
        return "Weak"
    return "Risk"


## Repository Feature Extraction

In [7]:
def parse_github_repo_url(repo_url_or_full_name):
    text = repo_url_or_full_name.strip()

    if text.startswith("http"):
        match = re.search(r"github\.com[:/]([^/]+)/([^/#?]+)", text)

        if not match:
            raise ValueError("Invalid GitHub repository URL")

        owner = match.group(1)
        repo = match.group(2).replace(".git", "")
        return f"{owner}/{repo}"

    if re.match(r"^[^/]+/[^/]+$", text):
        return text

    raise ValueError("Input must be a GitHub URL or owner/repo string")


def build_single_repo_features(repo_url_or_full_name):
    full_name = parse_github_repo_url(repo_url_or_full_name)
    raw_repo_df = build_repo_dataframe(full_name)
    feature_df = build_all_features(raw_repo_df)
    feature_df.insert(0, "repo_name", full_name)
    feature_df = add_engineered_features(feature_df)

    for feature in model_features:
        if feature not in feature_df.columns:
            feature_df[feature] = np.nan

    return full_name, raw_repo_df, feature_df


## Diagnosis Logic

In [8]:
def make_dimension_comment(score, dimension_key, detail):
    config = dimension_config[dimension_key]
    grade = score_to_grade(score)

    top_features = detail.sort_values("feature_score", ascending=False).head(3)
    bottom_features = detail.sort_values("feature_score", ascending=True).head(3)

    strengths = top_features["feature"].tolist()
    risks = bottom_features["feature"].tolist()

    if grade in ["Excellent", "Good"]:
        summary = f"{config['label_ko']} 점수는 {score:.1f}점으로 양호하다."
    elif grade == "Moderate":
        summary = f"{config['label_ko']} 점수는 {score:.1f}점으로 중간 수준이다."
    else:
        summary = f"{config['label_ko']} 점수는 {score:.1f}점으로 개선이 필요하다."

    return {
        "dimension": dimension_key,
        "dimension_label": config["label_ko"],
        "score": score,
        "grade": grade,
        "core_question": config["core_question"],
        "concepts": config["concepts"],
        "summary": summary,
        "strength_features": strengths,
        "risk_features": risks,
    }


def diagnose_repository(repo_url_or_full_name):
    full_name, raw_repo_df, repo_features = build_single_repo_features(repo_url_or_full_name)

    model_input = repo_features[model_features].copy()

    for col in model_features:
        model_input[col] = pd.to_numeric(model_input[col], errors="coerce")

    healthy_probability = float(model.predict_proba(model_input)[0, 1])
    overall_score = healthy_probability * 100

    dimension_rows = []
    detail_tables = []

    for dimension_key in dimension_config:
        score, detail = score_dimension(repo_features, dimension_key)
        dimension_rows.append(make_dimension_comment(score, dimension_key, detail))
        detail_tables.append(detail)

    dimension_summary = pd.DataFrame(dimension_rows)
    dimension_detail = pd.concat(detail_tables, ignore_index=True)

    diagnosis = {
        "repo_name": full_name,
        "overall_score": overall_score,
        "healthy_probability": healthy_probability,
        "overall_grade": score_to_grade(overall_score),
        "model_name": model_metadata.get("best_model_name"),
        "target": model_metadata.get("target_col"),
    }

    return diagnosis, dimension_summary, dimension_detail, repo_features, raw_repo_df


## Run Diagnosis

아래 `REPO_URL` 값을 원하는 GitHub repository URL 또는 `owner/repo` 형식으로 바꿔 실행한다.

In [9]:
REPO_URL = "https://github.com/pandas-dev/pandas"

diagnosis, dimension_summary, dimension_detail, repo_features, raw_repo_df = diagnose_repository(REPO_URL)

print("Repository:", diagnosis["repo_name"])
print("Overall OSS Health Score:", round(diagnosis["overall_score"], 2))
print("Healthy Probability:", round(diagnosis["healthy_probability"], 4))
print("Overall Grade:", diagnosis["overall_grade"])
print("Model:", diagnosis["model_name"])

dimension_summary[[
    "dimension_label",
    "score",
    "grade",
    "core_question",
    "summary",
    "strength_features",
    "risk_features",
]]


Repository: pandas-dev/pandas
Overall OSS Health Score: 99.75
Healthy Probability: 0.9975
Overall Grade: Excellent
Model: LogisticRegression


,dimension_label,score,grade,core_question,summary,strength_features,risk_features
0,커뮤니티 활성도,86.218765,Excellent,이 프로젝트는 현재 살아 움직이고 있는가?,커뮤니티 활성도 점수는 86.2점으로 양호하다.,"[num_events, has_PullRequestEvent, has_IssueCo...","[recent_event_density, development_ratio, non_..."
1,지속 가능성,74.838486,Good,이 프로젝트는 앞으로도 유지될 수 있는가?,지속 가능성 점수는 74.8점으로 양호하다.,"[num_contributors, is_recently_updated_90d, is...","[is_stale_365d, contributors_to_stars_ratio, l..."
2,코드 품질 및 신뢰성,44.964128,Weak,이 프로젝트의 산출물은 믿을 수 있는가?,코드 품질 및 신뢰성 점수는 45.0점으로 개선이 필요하다.,"[semver_tag_ratio, has_pull_requests, PullRequ...","[latest_tag_is_prerelease, issue_burden_score,..."
3,법적/운영 거버넌스,54.961752,Weak,이 프로젝트는 조직적으로 안전하게 운영되는가?,법적/운영 거버넌스 점수는 55.0점으로 개선이 필요하다.,"[has_issues, has_projects, has_downloads]","[disabled, archived, negative_repo_state]"
4,프로젝트 성숙도,69.073811,Moderate,이 프로젝트는 성숙한 운영 체계를 갖추었는가?,프로젝트 성숙도 점수는 69.1점으로 중간 수준이다.,"[num_tags, has_deployments, tag_based_deployme...","[release_quality_score, stars_per_size, tag_re..."


## Dimension Detail

In [10]:
dimension_detail.sort_values(["dimension_label", "feature_score"], ascending=[True, False]).head(100)


,dimension,dimension_label,feature,raw_value,higher_is_better,feature_score,weight
55,legal_operational_governance,법적/운영 거버넌스,has_issues,1.000000,True,100.000000,0.051197
56,legal_operational_governance,법적/운영 거버넌스,has_projects,1.000000,True,100.000000,0.231431
57,legal_operational_governance,법적/운영 거버넌스,has_downloads,1.000000,True,100.000000,0.004307
61,legal_operational_governance,법적/운영 거버넌스,allow_forking,1.000000,True,100.000000,0.000000
62,legal_operational_governance,법적/운영 거버넌스,has_pull_requests,1.000000,True,100.000000,0.017215
59,legal_operational_governance,법적/운영 거버넌스,has_pages,0.000000,True,73.594132,0.218320
58,legal_operational_governance,법적/운영 거버넌스,has_wiki,0.000000,True,59.413203,0.129945
60,legal_operational_governance,법적/운영 거버넌스,has_discussions,0.000000,True,50.366748,0.202742
63,legal_operational_governance,법적/운영 거버넌스,governance_openness_score,0.600000,True,46.699267,0.259418
64,legal_operational_governance,법적/운영 거버넌스,archived,0.000000,False,8.801956,0.149406


## Save Diagnosis Result

In [11]:
safe_repo_name = diagnosis["repo_name"].replace("/", "__")

summary_path = OUTPUT_DIR / f"{safe_repo_name}_dimension_summary.csv"
detail_path = OUTPUT_DIR / f"{safe_repo_name}_dimension_detail.csv"
feature_path = OUTPUT_DIR / f"{safe_repo_name}_features.csv"
json_path = OUTPUT_DIR / f"{safe_repo_name}_diagnosis.json"

dimension_summary.to_csv(summary_path, index=False)
dimension_detail.to_csv(detail_path, index=False)
repo_features.to_csv(feature_path, index=False)

json_ready = {
    "diagnosis": diagnosis,
    "dimensions": dimension_summary.to_dict("records"),
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_ready, f, ensure_ascii=False, indent=2)

print("Saved summary:", summary_path)
print("Saved detail:", detail_path)
print("Saved features:", feature_path)
print("Saved json:", json_path)


Saved summary: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/outputs/3_repository_diagnosis/pandas-dev__pandas_dimension_summary.csv
Saved detail: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/outputs/3_repository_diagnosis/pandas-dev__pandas_dimension_detail.csv
Saved features: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/outputs/3_repository_diagnosis/pandas-dev__pandas_features.csv
Saved json: /Users/carolyn/Desktop/3-1/opensource/data/oss-health-data/src/outputs/3_repository_diagnosis/pandas-dev__pandas_diagnosis.json


## Backend Function Shape

Backend에서는 아래 흐름을 API endpoint 내부에서 사용하면 된다.

1. request body에서 GitHub repository URL을 받는다.
2. `diagnose_repository(repo_url)`을 호출한다.
3. `overall_score`, `healthy_probability`, `dimension_summary`를 JSON으로 반환한다.

주의할 점은 GitHub API rate limit이다. 운영 환경에서는 `GITHUB_TOKEN` 환경 변수를 설정하고, 동일 repository에 대한 feature extraction 결과를 cache하는 것이 좋다.

In [12]:
def backend_response_example(repo_url_or_full_name):
    diagnosis, dimension_summary, dimension_detail, repo_features, raw_repo_df = diagnose_repository(
        repo_url_or_full_name
    )

    return {
        "repo_name": diagnosis["repo_name"],
        "overall_score": round(diagnosis["overall_score"], 2),
        "healthy_probability": round(diagnosis["healthy_probability"], 4),
        "overall_grade": diagnosis["overall_grade"],
        "dimension_scores": [
            {
                "dimension": row["dimension"],
                "label": row["dimension_label"],
                "score": round(row["score"], 2),
                "grade": row["grade"],
                "summary": row["summary"],
                "strength_features": row["strength_features"],
                "risk_features": row["risk_features"],
            }
            for row in dimension_summary.to_dict("records")
        ],
    }


backend_response_example(REPO_URL)


{'repo_name': 'pandas-dev/pandas',
 'overall_score': 99.75,
 'healthy_probability': 0.9975,
 'overall_grade': 'Excellent',
 'dimension_scores': [{'dimension': 'community_activity',
   'label': '커뮤니티 활성도',
   'score': 86.22,
   'grade': 'Excellent',
   'summary': '커뮤니티 활성도 점수는 86.2점으로 양호하다.',
   'strength_features': ['num_events',
    'has_PullRequestEvent',
    'has_IssueCommentEvent'],
   'risk_features': ['recent_event_density',
    'development_ratio',
    'non_external_activity_ratio']},
  {'dimension': 'sustainability',
   'label': '지속 가능성',
   'score': 74.84,
   'grade': 'Good',
   'summary': '지속 가능성 점수는 74.8점으로 양호하다.',
   'strength_features': ['num_contributors',
    'is_recently_updated_90d',
    'is_recently_pushed_30d'],
   'risk_features': ['is_stale_365d',
    'contributors_to_stars_ratio',
    'last_update_recency_days']},
  {'dimension': 'code_quality_reliability',
   'label': '코드 품질 및 신뢰성',
   'score': 44.96,
   'grade': 'Weak',
   'summary': '코드 품질 및 신뢰성 점수는 45.0점으로 개선이

## Interpretation Notes

총괄 점수는 최종 모델이 `new_label = 1`로 판단할 확률을 0~100점으로 변환한 값이다. 이 값은 5개 차원 점수의 단순 평균이 아니라, 최종 모델이 전체 feature pattern을 보고 계산한 probability score이다.

5개 차원 점수는 reference dataset 대비 percentile score이다. 따라서 특정 repository가 학습 기준 repository들과 비교했을 때 각 평가 차원에서 상대적으로 어느 위치에 있는지 보여준다.

현재 feature set에는 license text, SPDX license validation, vulnerability advisory, CI pass rate, test coverage, code review quality 같은 정보가 직접 포함되어 있지 않다. 따라서 `코드 품질 및 신뢰성`, `법적/운영 거버넌스`의 일부 항목은 GitHub metadata와 release/issue/governance proxy feature로 평가한다. 추후 GitHub Actions, license API, security advisory, dependency file 분석 feature를 추가하면 이 두 차원의 설명력이 더 좋아질 수 있다.

---

## Final Summary

본 노트북은 특정 GitHub repository URL을 입력받아 OSS health를 진단하는 최종 repository diagnosis pipeline이다.

`0_DATASET.ipynb`, `1_FEATURE.ipynb`, `2_MODEL.ipynb`를 통해 만든 dataset, feature engineering logic, 최종 모델을 활용하여 개별 repository의 상태를 점수화하고, 제안서에서 정의한 5가지 평가 차원별 분석 결과를 제공한다.

전체 흐름은 다음과 같다.

1. 저장된 최종 모델과 feature list를 불러온다.
2. GitHub repository URL 또는 `owner/repo` 형식의 입력을 받는다.
3. GitHub API를 통해 repository raw data를 수집한다.
4. 기존 feature extraction 함수를 사용해 raw feature를 생성한다.
5. 모델 학습 시 사용한 engineered feature를 동일하게 재생성한다.
6. 최종 모델의 `predict_proba` 결과를 총괄 OSS health score로 변환한다.
7. reference dataset 대비 percentile 방식으로 5개 평가 차원별 점수를 계산한다.
8. 각 차원별 강점 feature, 위험 feature, 해석 문장을 생성한다.
9. backend에서 사용할 수 있는 JSON 형태의 응답 예시를 만든다.
10. 진단 결과를 CSV와 JSON 파일로 저장한다.

---

### 1. 본 노트북의 목적

본 노트북의 목적은 학습된 OSS health model을 실제 repository 진단 로직으로 연결하는 것이다.

이전 노트북들의 역할은 다음과 같다.

- `0_DATASET.ipynb`
  - GitHub repository raw data 수집
  - feature extraction을 위한 dataset 생성
  - 초기 healthy / unhealthy repository 구성

- `1_FEATURE.ipynb`
  - feature family 정의
  - OSS health score 계산
  - `new_label` 생성
  - feature engineering
  - feature importance, SHAP, ablation 등 feature 분석

- `2_MODEL.ipynb`
  - `new_label` 기반 최종 modeling
  - 최적 feature set 탐색
  - 최적 model 선택
  - hyperparameter tuning
  - backend용 model artifact 저장

본 노트북은 위 과정을 통해 저장된 최종 모델을 사용하여, 새로운 repository 하나에 대해 실제 진단 결과를 생성한다.

---

### 2. 입력 형식

입력은 GitHub repository URL 또는 `owner/repo` 형식을 사용할 수 있다.

예시는 다음과 같다.

```text
https://github.com/pandas-dev/pandas
```

또는

```text
pandas-dev/pandas
```

노트북 내부에서는 입력 문자열을 parsing하여 GitHub API에서 사용할 수 있는 `owner/repo` 형식으로 변환한다.

---

### 3. 사용되는 저장 모델

본 노트북은 `2_MODEL.ipynb`에서 저장한 최종 모델 artifact를 불러온다.

사용하는 주요 파일은 다음과 같다.

```text
models/oss_health_best_model.joblib
models/oss_health_best_features.json
models/oss_health_model_metadata.json
```

각 파일의 역할은 다음과 같다.

#### `oss_health_best_model.joblib`

최종 학습된 sklearn-compatible pipeline이다.

이 pipeline은 preprocessing과 classifier를 포함하므로, backend에서도 동일하게 load하여 바로 prediction에 사용할 수 있다.

#### `oss_health_best_features.json`

최종 모델이 입력으로 받는 feature list이다.

새로운 repository에 대해 feature를 생성한 뒤, 반드시 이 feature 순서에 맞춰 model input을 구성해야 한다.

#### `oss_health_model_metadata.json`

모델 학습 관련 metadata를 포함한다.

예를 들어 다음 정보가 저장되어 있다.

- target column
- best model name
- selected feature list
- holdout metrics
- cross validation summary
- tuning parameter
- model 성능 정보

본 노트북에서는 이 metadata를 불러와 진단 결과와 함께 참고 정보로 사용한다.

---

### 4. Feature Extraction

새로운 repository에 대한 feature는 기존에 작성한 함수를 그대로 사용한다.

사용하는 함수는 다음과 같다.

```python
from data.extract_github_repo import build_repo_dataframe
from features.build_features import build_all_features
```

전체 흐름은 다음과 같다.

1. `build_repo_dataframe(full_name)`으로 GitHub API raw data를 수집한다.
2. `build_all_features(raw_repo_df)`로 raw feature를 생성한다.
3. `add_engineered_features(feature_df)`로 모델 학습 시 사용한 engineered feature를 재생성한다.
4. 최종 모델 feature list에 맞게 column을 정렬한다.
5. 누락된 feature가 있으면 `NaN`으로 채운다.
6. 저장된 model pipeline 내부의 imputer가 이를 처리한다.

이 구조를 사용하면 학습 시점의 feature logic과 inference 시점의 feature logic을 최대한 일치시킬 수 있다.

---

### 5. 총괄 OSS Health Score

총괄 점수는 최종 모델의 binary classification 결과가 아니라, threshold 적용 전 probability를 기반으로 계산한다.

모델은 다음 확률을 반환한다.

```python
healthy_probability = model.predict_proba(X)[0, 1]
```

이 값은 해당 repository가 `new_label = 1`, 즉 healthy repository로 분류될 확률이다.

본 노트북에서는 이 확률을 0~100점으로 변환하여 총괄 OSS health score로 사용한다.

```python
overall_score = healthy_probability * 100
```

따라서 총괄 점수의 의미는 다음과 같다.

```text
모델이 해당 repository를 healthy하다고 판단하는 확률 기반 점수
```

예를 들어 `healthy_probability = 0.82`라면 총괄 점수는 82점이다.

이 점수는 binary threshold 적용 전의 연속적인 health confidence score이므로, 단순히 healthy / unhealthy로 나누는 것보다 더 풍부한 해석이 가능하다.

---

### 6. 등급 체계

총괄 점수와 차원별 점수는 동일한 grade 체계로 해석한다.

등급 기준은 다음과 같다.

```text
85점 이상  -> Excellent
70점 이상  -> Good
55점 이상  -> Moderate
40점 이상  -> Weak
40점 미만  -> Risk
```

이 등급은 직관적인 해석을 돕기 위한 보조 기준이다.

실제 backend나 UI에서는 필요에 따라 threshold를 조정할 수 있다.

---

### 7. 5개 평가 차원

제안서에서 정의한 5가지 기준을 본 노트북의 진단 차원으로 사용한다.

평가 차원은 다음과 같다.

1. 커뮤니티 활성도
2. 지속 가능성
3. 코드 품질 및 신뢰성
4. 법적/운영 거버넌스
5. 프로젝트 성숙도

각 차원은 핵심 질문과 세부 개념을 가진다.

---

### 8. 커뮤니티 활성도

커뮤니티 활성도는 다음 질문에 답하기 위한 차원이다.

```text
이 프로젝트는 현재 살아 움직이고 있는가?
```

세부 개념은 다음과 같다.

```text
Activity Volume, Responsiveness, Engagement Quality
```

사용되는 주요 feature는 다음과 같다.

- `num_events`
- `num_unique_event_types`
- `event_type_entropy`
- `recent_event_density`
- `has_IssuesEvent`
- `has_PullRequestEvent`
- `has_IssueCommentEvent`
- `IssuesEvent_ratio`
- `IssueCommentEvent_ratio`
- `PullRequestEvent_ratio`
- `interaction_ratio`
- `development_ratio`
- `collaboration_event_score`
- `activity_diversity_score`
- `healthy_activity_score`
- `non_external_activity_ratio`

이 차원은 repository에서 issue, pull request, comment, push 등 실제 활동이 발생하는지 평가한다.

단순히 star나 fork가 많은 repository보다, 최근에도 다양한 event와 협업 활동이 발생하는 repository가 높은 점수를 받는다.

---

### 9. 지속 가능성

지속 가능성은 다음 질문에 답하기 위한 차원이다.

```text
이 프로젝트는 앞으로도 유지될 수 있는가?
```

세부 개념은 다음과 같다.

```text
Contributor Structure, Diversity, Activity Stability
```

사용되는 주요 feature는 다음과 같다.

- `num_contributors`
- `total_contributions`
- `median_contributions`
- `contribution_entropy`
- `top1_contribution_share`
- `top5_contribution_share`
- `contribution_gini`
- `top1_contribution_ratio`
- `top3_contribution_ratio`
- `bus_factor_risk`
- `distributed_contribution_score`
- `contributor_depth_score`
- `last_update_recency_days`
- `last_push_recency_days`
- `update_push_gap_days`
- `freshness_score`
- `maintainer_activity_score`

이 차원은 repository가 소수 contributor에게 과도하게 의존하는지, contributor base가 충분히 넓은지, 최근에도 유지보수가 이어지는지를 평가한다.

예를 들어 `top1_contribution_share`나 `contribution_gini`가 높으면 특정 contributor 의존도가 높아질 수 있으므로 risk signal로 해석한다.

반대로 `contribution_entropy`, `num_contributors`, `distributed_contribution_score`가 높으면 contributor 구조가 더 분산되어 있다고 볼 수 있다.

---

### 10. 코드 품질 및 신뢰성

코드 품질 및 신뢰성은 다음 질문에 답하기 위한 차원이다.

```text
이 프로젝트의 산출물은 믿을 수 있는가?
```

세부 개념은 다음과 같다.

```text
Engineering Practice, Defect Signals, Security Signals
```

사용되는 주요 feature는 다음과 같다.

- `semver_tag_ratio`
- `stable_tag_ratio`
- `latest_tag_is_stable`
- `latest_tag_is_prerelease`
- `prerelease_tag_ratio`
- `release_maturity_score`
- `release_quality_score`
- `has_pull_requests`
- `PullRequestEvent_ratio`
- `development_ratio`
- `open_issues_to_stars_ratio`
- `issues_per_size`
- `issue_burden_score`
- `compiled_ratio`
- `language_entropy`

현재 dataset에는 test coverage, CI status, static analysis result, vulnerability advisory, dependency risk와 같은 직접적인 코드 품질 feature가 포함되어 있지 않다.

따라서 본 노트북에서는 release practice, semantic versioning, stable release, issue burden, pull request activity 등을 proxy feature로 사용한다.

즉 이 차원의 점수는 실제 코드 품질을 직접 측정한다기보다, GitHub metadata에서 관찰 가능한 engineering reliability signal을 기반으로 계산된다.

---

### 11. 법적/운영 거버넌스

법적/운영 거버넌스는 다음 질문에 답하기 위한 차원이다.

```text
이 프로젝트는 조직적으로 안전하게 운영되는가?
```

세부 개념은 다음과 같다.

```text
Legal Compliance, Governance Structure
```

사용되는 주요 feature는 다음과 같다.

- `has_issues`
- `has_projects`
- `has_downloads`
- `has_wiki`
- `has_pages`
- `has_discussions`
- `allow_forking`
- `has_pull_requests`
- `governance_openness_score`
- `archived`
- `disabled`
- `negative_repo_state`

현재 dataset에는 license type, SPDX compliance, CLA 여부, security policy, code of conduct 같은 법적/운영 문서 feature가 직접 포함되어 있지 않다.

따라서 본 노트북에서는 repository의 운영 기능 활성화 여부와 archived / disabled 상태를 governance proxy로 사용한다.

특히 `archived`, `disabled`, `negative_repo_state`는 명확한 risk signal로 해석한다.

반대로 `has_issues`, `has_discussions`, `has_pull_requests`, `allow_forking` 등은 외부 참여와 운영 개방성을 나타내는 signal로 사용한다.

---

### 12. 프로젝트 성숙도

프로젝트 성숙도는 다음 질문에 답하기 위한 차원이다.

```text
이 프로젝트는 성숙한 운영 체계를 갖추었는가?
```

세부 개념은 다음과 같다.

```text
Release Engineering, Adoption/Popularity, Lifecycle/Scale
```

사용되는 주요 feature는 다음과 같다.

- `num_tags`
- `num_major_versions`
- `num_minor_versions`
- `tag_release_velocity`
- `num_deployments`
- `has_deployments`
- `num_unique_refs`
- `tag_based_deployment_ratio`
- `deployment_recency_days`
- `release_recency_score`
- `active_release_score`
- `release_quality_score`
- `stargazers_count`
- `subscribers_count`
- `forks_count`
- `network_count`
- `stars_per_repo_age_day`
- `forks_per_repo_age_day`
- `stars_per_size`
- `forks_per_size`
- `adoption_efficiency`
- `fork_interest_efficiency`
- `repo_age_days`
- `repo_size`

이 차원은 release 체계, versioning, deployment history, popularity, adoption, repository scale을 종합하여 프로젝트의 성숙도를 평가한다.

단순히 오래된 repository가 높은 점수를 받는 것이 아니라, release activity와 adoption signal이 함께 존재할 때 높은 점수를 받는다.

---

### 13. 차원별 점수 계산 방식

차원별 점수는 모델의 probability가 아니라 reference dataset 대비 percentile score로 계산한다.

계산 과정은 다음과 같다.

1. 각 차원에 해당하는 feature 목록을 정의한다.
2. 새로운 repository의 feature 값을 구한다.
3. 같은 feature에 대해 reference dataset의 분포를 가져온다.
4. repository 값이 reference dataset에서 어느 percentile에 위치하는지 계산한다.
5. 값이 클수록 좋은 feature는 percentile을 그대로 사용한다.
6. 값이 작을수록 좋은 feature는 `100 - percentile` 방식으로 뒤집는다.
7. 각 feature score를 가중 평균하여 dimension score를 만든다.

예를 들어 `num_events`는 값이 클수록 activity가 높다고 볼 수 있으므로 percentile을 그대로 사용한다.

반대로 `last_push_recency_days`는 값이 작을수록 최근에 push가 있었다는 뜻이므로 percentile을 뒤집어 사용한다.

---

### 14. Negative Direction Feature

일부 feature는 값이 클수록 risk signal이다.

이러한 feature는 차원 점수 계산 시 방향을 반대로 적용한다.

대표적인 negative direction feature는 다음과 같다.

- `top1_contribution_share`
- `top5_contribution_share`
- `contribution_gini`
- `top1_contribution_ratio`
- `top3_contribution_ratio`
- `bus_factor_risk`
- `deployment_recency_days`
- `prerelease_tag_ratio`
- `latest_tag_is_prerelease`
- `last_update_recency_days`
- `last_push_recency_days`
- `update_push_gap_days`
- `is_stale_365d`
- `open_issues_to_stars_ratio`
- `issues_per_size`
- `issue_burden_score`
- `dominant_event_ratio`
- `external_interest_event_ratio`
- `archived`
- `disabled`
- `negative_repo_state`

이 feature들은 값이 높을수록 score가 낮아지도록 처리한다.

---

### 15. Feature Weighting

차원별 점수 계산에서 모든 feature를 동일하게 취급하지 않고, 가능하면 모델 해석 결과를 반영한 weight를 사용한다.

사용 가능한 importance 파일은 다음과 같다.

- `final_model_importance.csv`
- `final_model_permutation_importance.csv`
- `shap_importance_result.csv`

이 파일들이 존재하면 각 feature의 importance를 읽어와 weight로 사용한다.

만약 importance 파일이 없거나 weight 합이 0이면 모든 feature에 동일한 weight를 적용한다.

이 방식은 차원 점수에도 최종 모델이 중요하게 본 feature의 영향이 반영되도록 하기 위한 것이다.

---

### 16. 차원별 해석 생성

각 차원에 대해 다음 정보를 생성한다.

- dimension key
- 한국어 차원명
- score
- grade
- 핵심 질문
- 세부 개념
- summary
- strength features
- risk features

`strength_features`는 해당 차원 안에서 feature score가 높은 상위 feature들이다.

`risk_features`는 해당 차원 안에서 feature score가 낮은 하위 feature들이다.

이를 통해 단순 점수뿐 아니라, 왜 해당 점수가 나왔는지 설명할 수 있다.

예를 들어 커뮤니티 활성도 점수가 낮다면 risk feature로 `num_events`, `IssueCommentEvent_ratio`, `PullRequestEvent_ratio` 등이 나타날 수 있다.

이 경우 해당 repository는 단순 popularity보다 실제 interaction이 부족하다고 해석할 수 있다.

---

### 17. 결과 저장

진단 결과는 다음 경로에 저장된다.

```text
outputs/3_repository_diagnosis/
```

저장되는 파일은 다음과 같다.

```text
{repo_name}_dimension_summary.csv
{repo_name}_dimension_detail.csv
{repo_name}_features.csv
{repo_name}_diagnosis.json
```

각 파일의 역할은 다음과 같다.

#### `dimension_summary.csv`

5개 평가 차원별 요약 결과를 저장한다.

포함 정보는 다음과 같다.

- dimension
- dimension label
- score
- grade
- core question
- concepts
- summary
- strength features
- risk features

#### `dimension_detail.csv`

각 dimension 안에서 feature별 점수와 raw value를 저장한다.

포함 정보는 다음과 같다.

- dimension
- feature
- raw value
- higher is better 여부
- feature score
- weight

#### `features.csv`

해당 repository에 대해 추출된 raw feature와 engineered feature를 저장한다.

#### `diagnosis.json`

backend 또는 frontend에서 바로 사용할 수 있는 JSON 형태의 진단 결과를 저장한다.

---

### 18. Backend 응답 구조

본 노트북에는 backend에서 사용할 수 있는 응답 형태를 예시로 제공한다.

응답 구조는 다음과 같다.

```json
{
  "repo_name": "owner/repo",
  "overall_score": 82.31,
  "healthy_probability": 0.8231,
  "overall_grade": "Good",
  "dimension_scores": [
    {
      "dimension": "community_activity",
      "label": "커뮤니티 활성도",
      "score": 78.4,
      "grade": "Good",
      "summary": "커뮤니티 활성도 점수는 78.4점으로 양호하다.",
      "strength_features": [],
      "risk_features": []
    }
  ]
}
```

Backend에서는 이 구조를 그대로 API response로 반환할 수 있다.

---

### 19. GitHub API 관련 주의사항

본 노트북은 새로운 repository를 진단할 때 GitHub API를 호출한다.

따라서 다음 사항에 주의해야 한다.

1. GitHub API rate limit이 발생할 수 있다.
2. `.env` 파일 또는 환경 변수에 `GITHUB_TOKEN`을 설정하는 것이 좋다.
3. 같은 repository를 반복 진단할 경우 raw API response 또는 feature 결과를 cache하는 것이 좋다.
4. API failure가 발생하면 일부 feature가 누락될 수 있다.
5. 누락된 feature는 `NaN`으로 채워지고, model pipeline 내부 imputer가 처리한다.

---

### 20. 현재 방식의 한계

현재 진단 로직은 GitHub API에서 수집 가능한 metadata와 activity feature를 기반으로 한다.

따라서 일부 평가 차원은 proxy feature에 의존한다.

특히 다음 정보는 현재 직접 반영되지 않는다.

- license text
- SPDX license validation
- security policy
- vulnerability advisory
- dependency vulnerability
- CI pass rate
- test coverage
- code review quality
- maintainer response time
- issue close time
- pull request merge time
- documentation quality
- governance document
- code of conduct
- contributor license agreement

이러한 feature를 추가하면 다음 차원의 정확도가 향상될 수 있다.

- 코드 품질 및 신뢰성
- 법적/운영 거버넌스
- 지속 가능성
- 커뮤니티 활성도

---

### 21. 향후 개선 방향

향후 개선 방향은 다음과 같다.

#### GitHub Actions 기반 CI feature 추가

- workflow 존재 여부
- 최근 workflow 성공률
- failed workflow 비율
- CI activity recency

#### Issue / Pull Request lifecycle feature 추가

- 평균 issue close time
- 평균 PR merge time
- stale issue ratio
- maintainer response time
- first response time

#### Security feature 추가

- security policy 존재 여부
- dependabot 활성화 여부
- vulnerability alert 여부
- release signing 여부

#### Legal / Governance feature 추가

- license 존재 여부
- SPDX-compatible license 여부
- code of conduct 존재 여부
- contributing guide 존재 여부
- governance document 존재 여부

#### Documentation feature 추가

- README 존재 여부
- docs directory 존재 여부
- examples directory 존재 여부
- API documentation 존재 여부

이러한 feature를 추가하면 제안서의 5개 평가 차원을 더 직접적으로 측정할 수 있다.

---

### 22. 본 노트북의 최종 역할

본 노트북은 최종 사용자 관점의 OSS repository 진단 알고리즘을 구현한다.

즉 기존 노트북들이 dataset 생성, feature engineering, model training을 담당했다면, 본 노트북은 학습된 모델을 실제 repository 평가에 적용한다.

최종 산출물은 다음과 같다.

- repository별 총괄 OSS health score
- 5개 평가 차원별 score
- 차원별 등급
- 차원별 강점 feature
- 차원별 위험 feature
- backend API response 형태
- 저장 가능한 diagnosis JSON

이 구조를 기반으로 frontend에서는 repository URL 입력 후 health score dashboard를 구성할 수 있고, backend에서는 저장된 모델과 feature extraction logic을 사용해 실시간 repository 진단 API를 구현할 수 있다.